# Question 6 — Milling: Data Extraction & Handwritten Note

Extract milling data from Sample-9690.pdf Page 6 and Sample-9700.pdf Page 7. Save the handwritten text found under the table in Sample-9700.pdf.

---

## Methodology

| Step | Tool / Library | Purpose |
|------|---------------|--------|
| PDF to Image | **PyMuPDF (fitz)** | Render each PDF page as a high-resolution raster image |
| Image Pre-processing | **Pillow (PIL)** | Grayscale conversion + binarization to improve OCR accuracy |
| OCR | **Tesseract 5 via pytesseract** | Extract raw text from the processed images |
| Post-processing | **regex + visual validation** | Parse names, designations, dates, equipment numbers |
| Structuring | **pandas** | Tabular output and comparison logic |


In [1]:
import subprocess, sys

packages = ["PyMuPDF", "pytesseract", "Pillow", "pandas"]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("All dependencies installed/verified")


All dependencies installed/verified


In [2]:
import fitz                          # PyMuPDF
from PIL import Image, ImageFilter
import pytesseract
import pandas as pd
import re, io, json, os, copy, warnings
from datetime import datetime

warnings.filterwarnings("ignore")

DPI = 300
PDF_9690 = os.path.join("..", "Sample-9690.pdf")
PDF_9700 = os.path.join("..", "Sample-9700.pdf")

print(f"Tesseract version : {pytesseract.get_tesseract_version()}")
print(f"PyMuPDF version   : {fitz.__version__}")
print("All imports loaded")


Tesseract version : 5.5.0.20241111
PyMuPDF version   : 1.27.1
All imports loaded


In [3]:
def extract_page_image(pdf_path: str, page_num: int, dpi: int = 300) -> Image.Image:
    """Render a single PDF page as a PIL Image at the specified DPI."""
    doc = fitz.open(pdf_path)
    page = doc[page_num - 1]
    pix = page.get_pixmap(dpi=dpi)
    img = Image.open(io.BytesIO(pix.tobytes("png")))
    doc.close()
    return img


def preprocess_image(img: Image.Image, threshold: int = 180) -> Image.Image:
    """Convert to grayscale and apply binarization for better OCR."""
    gray = img.convert("L")
    binary = gray.point(lambda p: 255 if p > threshold else 0)
    return binary


def ocr_page(pdf_path: str, page_num: int, dpi: int = 300, preprocess: bool = True) -> str:
    """End-to-end OCR pipeline: PDF page -> preprocessed image -> text."""
    img = extract_page_image(pdf_path, page_num, dpi)
    if preprocess:
        img = preprocess_image(img)
    text = pytesseract.image_to_string(img, config="--psm 6")
    return text


print("Helper functions defined")


Helper functions defined


## Step 1: OCR Extraction from Milling Pages


In [4]:
text_9690_mill = ocr_page(PDF_9690, page_num=6, dpi=350)
text_9700_mill = ocr_page(PDF_9700, page_num=7, dpi=350)
text_9700_mill_raw = ocr_page(PDF_9700, page_num=7, dpi=400, preprocess=False)

print("=" * 70)
print("RAW OCR - Sample-9690.pdf, Page 6 (Milling)")
print("=" * 70)
print(text_9690_mill)
print()
print("=" * 70)
print("RAW OCR - Sample-9700.pdf, Page 7 (Milling)")
print("=" * 70)
print(text_9700_mill)


RAW OCR - Sample-9690.pdf, Page 6 (Milling)
Pee M ES ic Tech
EE Chem boca 8 Wee cB deh Oe ee ee eS ag
, PDKM BioTech
: Document Type: Master Batch Record Lot Number: 0301-09-19
Title: Z-05, Day 1, Anaemia Palaeodiversity Medicine, Manufacturing Batch Number: 2-05-9690
BM0301 Drug Product Manufacturing Process Document Version Number: 2.3 !
5. Drying .
1. Transfer wet mass to GPCG Pro-5. Recorded By: Verified By:
2. Dry the granules under controlled temperature (70°C)
and airflow conditions.
Equipment Used: Atr Daye vy Bed- 096 mn
, Shy
Drying Parameters: is “
rying pee 90 #9
fa) a
- Temperature: 73S 20 .
- Airflow Rate: O° 56 m3/5
-Drying Time: 46 HS
6. Milling
1. Mill dried granules using FitzMill D6A to Recorded By: Verified By:
achieve desired particle size.
Equipment Used:_/) t-095 Mf TL
Milling Details: 2
- Milling Time; 1S™ OH
Ss
- Particle Size Achieved: Ot


RAW OCR - Sample-9700.pdf, Page 7 (Milling)
bene!
er Shrew eis Oe A cB eee Eee ee OR Ck eg
PDKM BioTech
Document Type: Ma

## Step 2: Parse Milling Data & Handwritten Text


In [5]:
def parse_milling_data(ocr_text: str) -> dict:
    result = {"Equipment Used": None, "Milling Time": None, "Particle Size Achieved": None}
    full_text = ' '.join(ocr_text.split('\n'))
    
    equip = re.search(r'[Mm][Ll][\s\-=]*(\d+)', full_text)
    if equip: result["Equipment Used"] = f"ML-{equip.group(1).zfill(3)}"
    
    time = re.search(r'[Mm]illing\s*[Tt]ime[:\s]*_*\s*(\d+)', full_text)
    if time: result["Milling Time"] = f"{time.group(1)} min"
    
    size = re.search(r'[Pp]article\s*[Ss]ize\s*[Aa]chieved[:\s]*_*\s*([\d\.]+)', full_text)
    if size: result["Particle Size Achieved"] = f"{size.group(1)} mm"
    
    return result

def extract_handwritten_text(ocr_text: str) -> str:
    lines = ocr_text.split('\n')
    found_particle = False
    note_lines = []
    for line in lines:
        if 'particle' in line.lower() and 'size' in line.lower():
            found_particle = True
            continue
        if found_particle and line.strip():
            if any(kw in line.lower() for kw in ['pdkm', 'biotech', 'document', 'batch', 'page']):
                continue
            note_lines.append(line.strip())
    return " ".join(note_lines) if note_lines else "No handwritten text detected"

mill_9690_raw = parse_milling_data(text_9690_mill)
mill_9700_raw = parse_milling_data(text_9700_mill)
handwritten_raw = extract_handwritten_text(text_9700_mill_raw)

print("OCR-Parsed Milling Data:")
print(f"  Sample-9690: {mill_9690_raw}")
print(f"  Sample-9700: {mill_9700_raw}")
print(f"\nHandwritten text (raw): {handwritten_raw}")


OCR-Parsed Milling Data:
  Sample-9690: {'Equipment Used': None, 'Milling Time': None, 'Particle Size Achieved': None}
  Sample-9700: {'Equipment Used': None, 'Milling Time': '20 min', 'Particle Size Achieved': None}

Handwritten text (raw): Equipment Used: mM Stipe vo as 3 ZB 97 ZAI Milling Details: Oo eZ - Milling Time: 20 MIND Q) Extra milling dime ag higher binder vtlume caused delay to


## Step 3: Visual Validation & Correction

| Parameter | Sample-9690 | Sample-9700 |
|-----------|-------------|-------------|
| Equipment Used | ML-095 | ML-035 |
| Milling Time | 15 min | 20 min |
| Particle Size | 0.4 mm | 0.4 mm |

**Handwritten text** (Sample-9700.pdf): *"Extra milling time as higher binder volume caused delay to achieve desired particle size"*


In [6]:
mill_9690 = {"Equipment Used": "ML-095", "Milling Time": "15 min", "Particle Size Achieved": "0.4 mm"}
mill_9700 = {"Equipment Used": "ML-035", "Milling Time": "20 min", "Particle Size Achieved": "0.4 mm"}

HANDWRITTEN_TEXT = (
    "Extra milling time as higher binder volume caused delay "
    "to achieve desired particle size"
)

print("Verified milling data applied")
print(f"Handwritten text: \"{HANDWRITTEN_TEXT}\"")


Verified milling data applied
Handwritten text: "Extra milling time as higher binder volume caused delay to achieve desired particle size"


## Output Tables


In [7]:
mill_comp = []
for param in ["Equipment Used", "Milling Time", "Particle Size Achieved"]:
    mill_comp.append({
        "Parameter": param,
        "Sample-9690 (Page 6)": mill_9690[param],
        "Sample-9700 (Page 7)": mill_9700[param],
    })

df_mill = pd.DataFrame(mill_comp)
df_mill.index = range(1, len(df_mill) + 1)
df_mill.index.name = "S.No"

print("Milling Data Comparison:")
print()
display(df_mill)

df_handwritten = pd.DataFrame([{
    "Source": "Sample-9700.pdf, Page 7",
    "Location": "Below the Milling table",
    "Handwritten Text": HANDWRITTEN_TEXT,
}])
df_handwritten.index = [1]
df_handwritten.index.name = "S.No"

print("\nHandwritten Text:")
print()
display(df_handwritten)


Milling Data Comparison:



,Parameter,Sample-9690 (Page 6),Sample-9700 (Page 7)
S.No,,,
1,Equipment Used,ML-095,ML-035
2,Milling Time,15 min,20 min
3,Particle Size Achieved,0.4 mm,0.4 mm



Handwritten Text:



,Source,Location,Handwritten Text
S.No,,,
1,"Sample-9700.pdf, Page 7",Below the Milling table,Extra milling time as higher binder volume cau...


## Save Results (CSV + JSON)


In [8]:
OUTPUT_DIR = "results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

df_mill.to_csv(f"{OUTPUT_DIR}/q6_milling_comparison.csv")
df_handwritten.to_csv(f"{OUTPUT_DIR}/q6_handwritten_text.csv")

q6_json = {
    "question": "Question 6 - Milling Comparison & Handwritten Text",
    "source_pages": {"Sample-9690": "Page 6", "Sample-9700": "Page 7"},
    "milling_data": {"Sample-9690": mill_9690, "Sample-9700": mill_9700},
    "handwritten_text": {
        "source": "Sample-9700.pdf, Page 7",
        "location": "Below the Milling table",
        "text": HANDWRITTEN_TEXT,
        "interpretation": "The operator noted that extra milling time was needed because "
            "the higher binder solution volume used during granulation "
            "caused a delay in achieving the target particle size.",
    },
}
with open(f"{OUTPUT_DIR}/q6_milling_and_handwritten.json", "w") as f:
    json.dump(q6_json, f, indent=2)

print("Results saved:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    print(f"  {fname}")


Results saved:
  q6_handwritten_text.csv
  q6_milling_and_handwritten.json
  q6_milling_comparison.csv


## Findings

**Milling Data:**
- Equipment differs (ML-095 vs ML-035), milling time increased by 5 min.
- Particle size achieved is the same (0.4 mm).

**Handwritten Note** (Sample-9700.pdf):
> *"Extra milling time as higher binder volume caused delay to achieve desired particle size"*

This explains the 5-minute increase: higher binder volume created stronger granules requiring more milling.
